# Libraries & Paths

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import niqe

# Settings
INPUT_PATH = r"C:\Users\makvanmt1\Downloads\test\test\Images\Gridded_Images"
OUTPUT_FOLDER = r"C:\Users\makvanmt1\Downloads\test\test\Images\Gridded_Images\Interpolation_Results"

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

# Defaults
EDGE_THRESH_DEFAULT = 0.09 
NEIGHBOR_CNT_DEFAULT = 3   
INTERP_WD_DEFAULT = 9     
VALID_RATIO_DEFAULT = 0.8
valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tif')

# Image Loading

In [4]:
if os.path.isdir(INPUT_PATH):
    image_files = [f for f in os.listdir(INPUT_PATH) if f.lower().endswith(valid_extensions)]
    if not image_files:
        raise FileNotFoundError(f"No images found in folder: {INPUT_PATH}")
    preview_file_path = os.path.join(INPUT_PATH, image_files[0])
else:
    preview_file_path = INPUT_PATH

static_image_bgr = cv2.imread(preview_file_path)
if static_image_bgr is None:
    raise ValueError(f"Could not load image at {preview_file_path}")

# Core Functions

In [5]:
def numpy_to_jpeg(array):
    if array.dtype != np.uint8:
        array = (array * 255).astype(np.uint8) if array.max() <= 1.0 else array.astype(np.uint8)
    if len(array.shape) == 2:
        array = cv2.cvtColor(array, cv2.COLOR_GRAY2BGR)
    _, encoded_image = cv2.imencode('.jpg', array, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    return encoded_image.tobytes()

def run_pipeline_on_frame(gray_frame, edge_t, ncnt, wd):
    wd = int(wd)
    if wd % 2 == 0: wd += 1
    
    inv = cv2.bitwise_not(gray_frame)
    _, binary = cv2.threshold(inv, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    B = (binary // 255).astype(np.uint8)
    lap = cv2.Laplacian(inv, cv2.CV_32F, ksize=3)
    mag = np.abs(lap)
    mag /= (mag.max() + 1e-6)
    neighbor_count = cv2.filter2D(B.astype(np.float32), -1, np.ones((3,3), np.uint8))
    keep = (B == 1) & (neighbor_count >= ncnt) & (mag >= edge_t)
    mask = np.zeros_like(B, dtype=np.uint8)
    mask[keep] = 255
    
    kernel = np.ones((wd, wd), np.float32)
    valid_count = cv2.filter2D((1 - (mask//255)).astype(np.float32), -1, kernel)
    dense_valid = (valid_count >= VALID_RATIO_DEFAULT * (wd * wd))
    mask_refined = np.where(dense_valid, 0, (mask//255)).astype(np.uint8) * 255
    
    out = gray_frame.copy()
    h = wd // 2
    for r, c in np.argwhere(mask_refined == 255):
        if r >= h and r < gray_frame.shape[0]-h and c >= h and c < gray_frame.shape[1]-h:
            hood = gray_frame[r-h:r+h+1, c-h:c+h+1]
            m_hood = mask_refined[r-h:r+h+1, c-h:c+h+1]
            valid_pix = hood[m_hood == 0]
            out[r, c] = np.mean(valid_pix) if valid_pix.size > 0 else np.mean(hood)
            
    blurred = cv2.GaussianBlur(out, (0, 0), 1.0)
    return cv2.addWeighted(out, 2.5, blurred, -1.5, 0)

def get_safe_niqe(img):
    h, w = img.shape
    if h < 193 or w < 193:
        scale = 193 / min(h, w)
        img = cv2.resize(img, (int(w * scale) + 1, int(h * scale) + 1), interpolation=cv2.INTER_CUBIC)
    return niqe.niqe(img.astype(np.float64))

# UI Elements

In [6]:
# Sliders
edge_thresh_slider = widgets.FloatSlider(value=EDGE_THRESH_DEFAULT, min=0.0, max=0.2, step=0.005, description='Edge Thresh:', continuous_update=False)
neighbor_cnt_slider = widgets.IntSlider(value=NEIGHBOR_CNT_DEFAULT, min=0, max=9, description='Neighbor Cnt:', continuous_update=False)
interp_wd_slider = widgets.IntSlider(value=INTERP_WD_DEFAULT, min=3, max=51, step=2, description='Window Size:', continuous_update=False)

# Displays
original_img_w = widgets.Image(format='jpeg', width=450)
processed_img_w = widgets.Image(format='jpeg', width=450)
debug_view = widgets.Label(value="Status: Ready")
progress_output = widgets.Output()

def update_preview(change=None):
    debug_view.value = f"Status: Processing... (Edge: {edge_thresh_slider.value})"
    gray = cv2.cvtColor(static_image_bgr, cv2.COLOR_BGR2GRAY)
    proc = run_pipeline_on_frame(gray, edge_thresh_slider.value, neighbor_cnt_slider.value, interp_wd_slider.value)
    original_img_w.value = numpy_to_jpeg(static_image_bgr)
    processed_img_w.value = numpy_to_jpeg(proc)
    debug_view.value = "Status: Updated"

def run_folder_process(b):
    folder = INPUT_PATH if os.path.isdir(INPUT_PATH) else os.path.dirname(INPUT_PATH)
    files = [f for f in os.listdir(folder) if f.lower().endswith(valid_extensions)]
    csv_path = os.path.join(OUTPUT_FOLDER, "niqe_batch_results.csv")
    
    with progress_output:
        clear_output()
        print(f"🚀 Processing {len(files)} images...")
        results = []
        for i, f in enumerate(files):
            path = os.path.join(folder, f)
            img = cv2.imread(path)
            if img is None: continue
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            proc = run_pipeline_on_frame(gray, edge_thresh_slider.value, neighbor_cnt_slider.value, interp_wd_slider.value)
            cv2.imwrite(os.path.join(OUTPUT_FOLDER, f"processed_{f}"), proc)
            s_in, s_out = get_safe_niqe(gray), get_safe_niqe(proc)
            results.append({"Filename": f, "Input NIQE": round(s_in, 3), "Output NIQE": round(s_out, 3), "Gain": round(s_in - s_out, 3)})
            if (i+1) % 5 == 0: print(f"Done {i+1}/{len(files)}...")

        df = pd.DataFrame(results)
        df.to_csv(csv_path, index=False)
        display(df)
        print(f"\nReport saved to: {csv_path}")

# Layout & Render

In [7]:
edge_thresh_slider.observe(update_preview, names='value')
neighbor_cnt_slider.observe(update_preview, names='value')
interp_wd_slider.observe(update_preview, names='value')

folder_btn = widgets.Button(description="Process All & Save CSV", button_style='danger', layout=widgets.Layout(width='250px'))
folder_btn.on_click(run_folder_process)

ui = widgets.VBox([
    widgets.HBox([
        widgets.VBox([widgets.HTML("<b>Original</b>"), original_img_w]), 
        widgets.VBox([widgets.HTML("<b>Processed Preview</b>"), processed_img_w])
    ]),
    widgets.VBox([
        debug_view,
        widgets.HTML("<br><b>Adjust Parameters (Preview updates on release):</b>"),
        edge_thresh_slider, neighbor_cnt_slider, interp_wd_slider,
        widgets.HTML("<br>"),
        folder_btn,
        progress_output
    ])
])

update_preview()
display(ui)